In [38]:
from langgraph.checkpoint.memory import MemorySaver
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langchain_core.messages import HumanMessage

In [39]:
from dotenv import load_dotenv

load_dotenv()

llm = ChatGroq(
    model="llama-3.3-70b-versatile"
)

In [40]:
class JokeState(TypedDict):
  topic:str
  joke:str
  explanation:str

In [41]:
def generate_joke(state: JokeState):

    topic = state["topic"]

    response = llm.invoke(
        [
            HumanMessage(
                content=f"Generate a short funny joke about {topic}. Only return the joke."
            )
        ]
    )

    return {
        "joke": response.content
    }


def explain_joke(state: JokeState):

    joke = state["joke"]

    response = llm.invoke(
        [
            HumanMessage(
                content=f"Explain this joke in simple words:\n{joke}"
            )
        ]
    )

    return {
        "explanation": response.content
    }

In [42]:
builder = StateGraph(JokeState)

builder.add_node("generate_joke", generate_joke)
builder.add_node("explain_joke", explain_joke)

builder.add_edge(START, "generate_joke")
builder.add_edge("generate_joke", "explain_joke")
builder.add_edge("explain_joke", END)


memory = MemorySaver()

graph = builder.compile(
    checkpointer=memory
)

config = {
    "configurable": {
        "thread_id": "2"
    }
}

In [43]:
result = graph.invoke(
    {
        "topic": "ai"
    },
    config=config
)

print("\nJoke:")
print(result["joke"])

print("\nExplanation:")
print(result["explanation"])


current_state = graph.get_state(config)

print(current_state.values)


history = list(graph.get_state_history(config))

for checkpoint in history:
    print(checkpoint)


Joke:
Why did the AI program go on a diet? Because it wanted to lose some bytes.

Explanation:
This joke is funny because it plays with the word "bytes". 

In computers, a "byte" is a unit of digital information. But the word "bytes" sounds similar to "pounds" or "weight", which is what people try to lose when they go on a diet.

So, the joke is saying that the AI program went on a diet to lose some "bytes" (like computer information), but it sounds like it's trying to lose weight, like a person would. It's a clever play on words.
{'topic': 'ai', 'joke': 'Why did the AI program go on a diet? Because it wanted to lose some bytes.', 'explanation': 'This joke is funny because it plays with the word "bytes". \n\nIn computers, a "byte" is a unit of digital information. But the word "bytes" sounds similar to "pounds" or "weight", which is what people try to lose when they go on a diet.\n\nSo, the joke is saying that the AI program went on a diet to lose some "bytes" (like computer informati